# Stock Movement Context Dashboard: Cleaning and Validation

This notebook explains the inspection and preparation of the prices, news, and filings tables. It runs `cleaning_scripts.py` from the original Parquet inputs and presents the checks, reasons, results, and remaining decisions.

**Reproducibility:** use Python 3.10 or later, install `requirements.txt`, and run all cells from the project folder. The source files in `data/` are preserved.


## CRISP-DM: Business understanding

The goal is to help an investor-relations analyst review unusually wide daily stock movements with nearby company news and the latest available 10-K or 10-Q. These sources may suggest context for human follow-up but cannot establish causation.

The project description mentions raw daily range (`high - low`) and gives a top-5% example using normalized range (`(high - low) / close`). This workflow calculates both, but does not choose a flagging threshold or label days as flagged.


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_DIR = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "cleaning_scripts.py").is_file())
sys.path.insert(0, str(PROJECT_DIR))
from cleaning_scripts import run_cleaning
report = run_cleaning(PROJECT_DIR, verbose=False)
print("Pipeline ran from the original Parquet inputs.")
for name, details in report["tables"].items():
    print(f"{name}: {details['rows']} source rows -> {details['rows_after']} output rows")
print(f"Five record checks passed: {report['record_check_pass_count']}/{report['record_check_count']}")


Pipeline ran from the original Parquet inputs.
prices: 2656 source rows -> 2656 output rows
news: 3888 source rows -> 3888 output rows
filings: 73 source rows -> 73 output rows
Five record checks passed: 5/5


## CRISP-DM: Data understanding and preparation

**Why check types?** Dates are used for sorting, context windows, and as-of filing selection. Source dates load as object/text, so the script parses them and counts failures. Invalid dates, if any, remain as `NaT` for review instead of being silently dropped. IDs stay identifiers.

**Why preserve rows and text?** No missing or duplicate rows currently require removal. News includes calendar dates without a matching trading row, which are useful for nearby context. Filing text is preserved exactly; although filing text has boundary whitespace, trimming it has not been approved.


In [2]:
for name, details in report["tables"].items():
    print(f"\n{name.upper()} — {details['rows']} rows")
    print("Types before:", details["dtypes_before"])
    print("Types after:", details["dtypes_after"])
    print("Date conversion failures:", details["date_conversion_failures"])
    print("Nulls before:", details["nulls_by_column"])
    print("Exact duplicates / duplicate IDs / candidate-key duplicates:", details["exact_duplicate_rows"], details["duplicate_ids"], details["duplicate_candidate_keys"])
    print("Rows before -> after:", details["rows"], "->", details["rows_after"])



PRICES — 2656 rows
Types before: {'id': 'int32', 'symbol': 'object', 'date': 'object', 'open': 'float64', 'high': 'float64', 'low': 'float64', 'close': 'float64', 'adj_close': 'float64', 'volume': 'int64'}
Types after: {'id': 'int32', 'symbol': 'object', 'date': 'datetime64[ns]', 'open': 'float64', 'high': 'float64', 'low': 'float64', 'close': 'float64', 'adj_close': 'float64', 'volume': 'int64'}
Date conversion failures: 0
Nulls before: {'id': 0, 'symbol': 0, 'date': 0, 'open': 0, 'high': 0, 'low': 0, 'close': 0, 'adj_close': 0, 'volume': 0}
Exact duplicates / duplicate IDs / candidate-key duplicates: 0 0 0
Rows before -> after: 2656 -> 2656

NEWS — 3888 rows
Types before: {'id': 'int32', 'symbol': 'object', 'date': 'object', 'highlights': 'object'}
Types after: {'id': 'int32', 'symbol': 'object', 'date': 'datetime64[ns]', 'highlights': 'object'}
Date conversion failures: 0
Nulls before: {'id': 0, 'symbol': 0, 'date': 0, 'highlights': 0}
Exact duplicates / duplicate IDs / candidate-k

## Prices: validity and movement calculations

**Why these checks?** Invalid OHLC relationships or nonpositive prices can make range calculations misleading. The script checks positive prices, `high >= low`, open and close within the daily low-high bounds, and nonnegative volume.

Raw range is retained for display; normalized range supports scale-adjusted comparison across companies. No threshold is applied until the flagging choice is confirmed.


In [3]:
price_profile = report["tables"]["prices"]
print("Price plausibility results:", price_profile["price_validity"])
print("Range formulas:", report["calculation_checks"]["definition_raw"], "and", report["calculation_checks"]["definition_normalized"])
print("Metric records:", report["calculation_checks"]["record_count"])
print("Metric records by symbol:", report["calculation_checks"]["record_count_by_symbol"])
print("Missing symbol group keys:", report["calculation_checks"]["missing_group_keys"])
print("Non-finite metric values:", report["calculation_checks"]["non_finite_metric_values"])
print("Missing metric values:", report["calculation_checks"]["missing_metric_values"])
print("Flag threshold applied:", report["calculation_checks"]["flag_threshold_applied"])
print("Normalized range summary by symbol:")
for symbol, stats in report["calculation_checks"]["normalized_range_by_symbol"].items(): print(symbol, stats)


Price plausibility results: {'nonpositive_by_column': {'open': 0, 'high': 0, 'low': 0, 'close': 0, 'adj_close': 0}, 'high_below_low': 0, 'open_outside_low_high': 0, 'close_outside_low_high': 0, 'negative_volume': 0}
Range formulas: high - low and (high - low) / close
Metric records: 2656
Metric records by symbol: {'AAPL': 332, 'ADBE': 332, 'AMZN': 332, 'GOOGL': 332, 'META': 332, 'MSFT': 332, 'NVDA': 332, 'TSLA': 332}
Missing symbol group keys: 0
Non-finite metric values: 0
Missing metric values: 0
Flag threshold applied: False
Normalized range summary by symbol:
AAPL {'record_count': 332, 'minimum': 0.006468478631796004, 'mean': 0.022010315784549157, 'maximum': 0.14443047693824218}
ADBE {'record_count': 332, 'minimum': 0.007822177193917096, 'mean': 0.02535111068535382, 'maximum': 0.0885531133385178}
AMZN {'record_count': 332, 'minimum': 0.005617019694280838, 'mean': 0.024008328393997455, 'maximum': 0.12569895851752738}
GOOGL {'record_count': 332, 'minimum': 0.008952609052126035, 'mean'

## News: calendar-day coverage and context window

Prices exist on trading days; news is one row per company per calendar day. Weekend and holiday news can still fall inside the context window, so unmatched news records are retained. Same-day coverage is checked separately from the seven-day window.


In [4]:
cross = report["cross_table_validation"]
print("Same-day price/news coverage:", cross["same_day_news_price"])
print("News context-window validation:", cross["news_context_window"])
print("Missing calendar dates by symbol:", cross["news_calendar_missing_dates_by_symbol"])


Same-day price/news coverage: {'news_rows_with_same_day_price': 2656, 'news_rows_without_same_day_price_retained': 1232, 'price_rows_with_same_day_news': 2656, 'relationship_for_same_day_join': 'one-to-one on (symbol, date) for current data', 'policy': 'Keep unmatched calendar-day news; do not use an inner join that removes it.'}
News context-window validation: {'price_events_checked': 2656, 'window_definition': 'same symbol, from event date minus 7 calendar days through event date, inclusive', 'events_with_no_news': 0, 'minimum_news_records_per_event': 2, 'median_news_records_per_event': 8.0, 'maximum_news_records_per_event': 8}
Missing calendar dates by symbol: {'AAPL': 0, 'ADBE': 0, 'AMZN': 0, 'GOOGL': 0, 'META': 0, 'MSFT': 0, 'NVDA': 0, 'TSLA': 0}


## Filings: latest available filing

A filing published after a price movement cannot provide context available on that date. For each price date, the script selects the latest same-company filing on or before that date. The original filing text is preserved.


In [5]:
print("Filing document types:", report["tables"]["filings"]["document_type_counts"])
print("Filing text checks:", report["tables"]["filings"]["text_checks"])
print("As-of filing validation:", cross["latest_prior_filing"])


Filing document types: {'10-Q': 51, '10-K': 22}
Filing text checks: {'document_type': {'empty_or_whitespace_only': 0, 'leading_or_trailing_whitespace': 0, 'minimum_length': 4, 'median_length': 4.0, 'maximum_length': 4}, 'mda_content': {'empty_or_whitespace_only': 0, 'leading_or_trailing_whitespace': 73, 'minimum_length': 15930, 'median_length': 46697.0, 'maximum_length': 76907}, 'risk_content': {'empty_or_whitespace_only': 0, 'leading_or_trailing_whitespace': 73, 'minimum_length': 333, 'median_length': 66162.0, 'maximum_length': 196794}}
As-of filing validation: {'price_events_checked': 2656, 'events_with_prior_filing': 2656, 'events_without_prior_filing': 0, 'future_filings_selected': 0, 'relationship': 'many price dates to one latest prior filing per symbol/date'}


## CRISP-DM: Evaluation and record checks

Five record checks include an ordinary price row, an extreme range, calendar-day news without same-day price, weekend news within a context window, and a filing selected at the event-date boundary. Each check states the expected rule before comparing the actual result.


In [6]:
checks = pd.read_csv(PROJECT_DIR / "outputs" / "record_checks.csv")
for row in checks.itertuples(index=False):
    print(f"Check {row.check_id} — {row.table}: {row.record}")
    print("  Expected:", row.expected_before_check)
    print("  Actual:", row.actual)
    print("  Matches:", row.matches_expectation)


Check 1 — prices: AAPL 2024-12-02
  Expected: Source price values are unchanged; date is valid and unique by symbol/date.
  Actual: date=2024-12-02, values_preserved=True, symbol-date duplicates=0
  Matches: True
Check 2 — prices: TSLA 2025-04-09
  Expected: Derived range equals high minus low; normalized range equals range divided by close.
  Actual: daily_range=50.80999756, normalized_range=0.1866642001, formulas_match=True
  Matches: True
Check 3 — news: AAPL 2024-12-01
  Expected: Retain the news row and text even though no same-day trading price exists.
  Actual: retained=True, no_same_day_price=True, text_preserved=True
  Matches: True
Check 4 — news: AAPL 2024-12-07 in AAPL 2024-12-09 context
  Expected: AAPL weekend news is retained and falls inside the Dec 2–Dec 9 inclusive context window.
  Actual: weekend_row_retained=True, included_in_window=True, total_window_news=8
  Matches: True
Check 5 — filings: ADBE latest filing for price date 2026-03-31
  Expected: Select the lates

## CRISP-DM: Deployment and remaining decisions

The script writes cleaned tables, calculations and lookup files, five record checks, a 48-row difficult-case sample, and a JSON report under `outputs/`. See `README.md` for the command and expected files.

**Still for review:** choose the movement flagging threshold; decide whether to trim filing-text boundary whitespace; record the Hugging Face revision or download date if available; and fill in your individual branch, branch URL, and commit ID in the README after pushing. Broader market and macroeconomic factors may also be missing.


In [7]:
sample = pd.read_parquet(PROJECT_DIR / "outputs" / "data_sample.parquet")
print("Sample rows (maximum 50):", len(sample))
print("Rows by table:", sample["source_table"].value_counts().to_dict())
print("Selection reasons:", sample["sample_reason"].value_counts().to_dict())
print("Output files:")
for output in sorted((PROJECT_DIR / "outputs").iterdir()): print(" -", output.name)


Sample rows (maximum 50): 48
Rows by table: {'prices': 16, 'news': 16, 'filings': 16}
Selection reasons: {'lowest normalized daily range': 8, 'highest normalized daily range': 8, 'calendar-day news without same-day price': 8, 'news on a trading date with same-day price': 8, 'shortest risk text': 8, 'longest risk text': 8}
Output files:
 - cleaning_report.json
 - data_sample.csv
 - data_sample.parquet
 - filing_lookup.csv
 - filings_cleaned.parquet
 - news_cleaned.parquet
 - news_window_counts.csv
 - price_metrics.csv
 - prices_cleaned.parquet
 - record_checks.csv
